# Inferencia de Modelos para Clasificación y Detección de Duplicados en Documentos Empresariales

Este notebook tiene como objetivo **cargar y utilizar los modelos entrenados** en el marco de la investigación aplicada en **Procesamiento del Lenguaje Natural (PLN)** orientada a empresas guatemaltecas. Se integran dos componentes clave del sistema propuesto:

1. **Clasificación automática de documentos** mediante un modelo **BETO fine-tuned**, adaptado a las categorías relevantes del dominio empresarial local (por ejemplo: *Cotizaciones*, *Facturas*, *Memorandos*, *Correspondencia*, etc.).
2. **Detección de documentos duplicados o altamente similares** utilizando **Sentence-BERT**, comparando nuevos textos contra un corpus de referencia previamente anonimizado.

El flujo de inferencia está diseñado para:
- Probar modelos con documentos nuevos (texto plano).
- Validar la coherencia de las predicciones en escenarios reales.
- Apoyar la evaluación cualitativa con usuarios administrativos.
- Servir como base para una futura API integrable en aplicaciones web.

> **Nota**: Este notebook asume que ya se han generado los siguientes artefactos:
> - `modelo_beto_finetuned_v1/`: directorio con el modelo y tokenizer de BETO.
> - `modelo_sbert_final/`: directorio con el modelo Sentence-BERT fine-tuned o preentrenado.
> - `corpus_anon.csv`: corpus de documentos empresariales anonimizados, con columnas `filename` y `category`.
> - `embeddings_full_corpus.npy`: embeddings precomputados del corpus completo.

Los resultados de este notebook contribuyen directamente a la **validación empírica** del sistema propuesto, alineado con el enfoque **CRISP-DM** y el cierre de la brecha entre investigación académica en PLN y su aplicación práctica en contextos locales.

In [ ]:
# Solo ejecutar si estás en Colab o un entorno limpio
# !pip install -q "transformers>=4.36.0" "torch>=2.6" sentence-transformers scikit-learn pandas numpy fastapi

## Importaciones

Se cargan las bibliotecas necesarias para:
- Manipulación de datos (`pandas`, `numpy`)
- Gestión de rutas (`pathlib`)
- Carga y uso del modelo **BETO** para clasificación (`transformers`)
- Codificación de etiquetas (`LabelEncoder`)
- Uso de **Sentence-BERT** para representación de oraciones (`sentence-transformers`)
- Cálculo de similitud coseno (`sklearn`)
- Supresión de advertencias no críticas

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Para BETO
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.preprocessing import LabelEncoder

# Para Sentence-BERT
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Advertencias
import warnings
warnings.filterwarnings("ignore")

C:\Users\Rolando\entorno_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Rolando\entorno_gpu\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## Configuración de rutas y verificación de artefactos

Se definen las rutas a los artefactos generados durante el entrenamiento y preprocesamiento:
- Modelo BETO fine-tuned (`modelo_beto_finetuned_v1`)
- Modelo Sentence-BERT (`modelo_sbert_final`)
- Corpus anonimizado (`corpus_anon.csv`)
- Embeddings precomputados del corpus completo (`embeddings_full_corpus.npy`)

Se valida la existencia de cada archivo o directorio. Si alguno falta, el notebook interrumpe la ejecución con un mensaje claro.  
Si todo está presente, se muestra un mensaje de confirmación con las rutas utilizadas.

In [10]:
# Rutas a los modelos y datos generados en tu flujo principal
BASE_DIR = Path(".")

BETO_MODEL_PATH = BASE_DIR / "models" / "modelo_beto_finetuned_v1"
SBERT_MODEL_PATH = BASE_DIR / "modelo_sbert_final"
CORPUS_ANON_PATH = BASE_DIR / "data" / "corpus_anon.csv"
EMBEDDINGS_PATH = BASE_DIR / "embeddings_full_corpus.npy"

# Verificar existencia
assert BETO_MODEL_PATH.exists(), "❌ No se encontró el modelo BETO"
assert SBERT_MODEL_PATH.exists(), "❌ No se encontró el modelo Sentence-BERT"
assert CORPUS_ANON_PATH.exists(), "❌ No se encontró el corpus anonimizado"
assert EMBEDDINGS_PATH.exists(), "❌ No se encontraron los embeddings"

print("✅ Todas las rutas verificadas con éxito:")
print(f" - Modelo BETO: {BETO_MODEL_PATH}")
print(f" - Modelo Sentence-BERT: {SBERT_MODEL_PATH}")
print(f" - Corpus anonimizado: {CORPUS_ANON_PATH}")
print(f" - Embeddings precomputados: {EMBEDDINGS_PATH}")

✅ Todas las rutas verificadas con éxito:
 - Modelo BETO: models\modelo_beto_finetuned_v1
 - Modelo Sentence-BERT: modelo_sbert_final
 - Corpus anonimizado: data\corpus_anon.csv
 - Embeddings precomputados: embeddings_full_corpus.npy


## Carga del modelo BETO fine-tuned

Se cargan el **tokenizer** y el **modelo de clasificación** previamente entrenado (fine-tuned) sobre el corpus empresarial en español guatemalteco.  
A partir de la configuración del modelo se extraen las etiquetas de las categorías (`id2label`), lo que permite interpretar las predicciones numéricas como nombres de clase legibles (por ejemplo: *Cotizaciones*, *Facturas*, etc.).

Se muestra la lista de categorías reconocidas por el modelo para confirmar que la carga fue correcta.

In [3]:
# Cargar modelo y tokenizer de BETO
print("Cargando modelo BETO fine-tuned...")
tokenizer_beto = AutoTokenizer.from_pretrained(BETO_MODEL_PATH)
model_beto = AutoModelForSequenceClassification.from_pretrained(BETO_MODEL_PATH)

# Recuperar clases desde el modelo (id2label)
id2label = model_beto.config.id2label
label2id = {v: k for k, v in id2label.items()}
le = LabelEncoder()
le.classes_ = np.array([id2label[i] for i in sorted(id2label.keys())])

print("✅ Clases del modelo:", list(id2label.values()))

Cargando modelo BETO fine-tuned...
✅ Clases del modelo: ['Contratos', 'Correos electrónicos', 'Correspondencia administrativa', 'Cotizaciones', 'Documentos fiscales', 'Recursos Humanos']


In [4]:
print("Cargando Sentence-BERT y corpus de referencia...")

# Cargar modelo SBERT
sbert = SentenceTransformer(str(SBERT_MODEL_PATH))

# Cargar corpus anonimizado y embeddings
df_corpus = pd.read_csv(CORPUS_ANON_PATH)
embeddings = np.load(EMBEDDINGS_PATH)

# Verificar alineación
assert len(df_corpus) == embeddings.shape[0], "❌ Desalineación entre embeddings y corpus"

print(f"✅ Corpus cargado: {len(df_corpus)} documentos")
print(f"✅ Embeddings shape: {embeddings.shape}")

Cargando Sentence-BERT y corpus de referencia...


The tokenizer you are loading from 'modelo_sbert_final' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✅ Corpus cargado: 906 documentos
✅ Embeddings shape: (906, 768)


## Función de clasificación de documentos

Define la función `classify_document(text)`, que:
- Preprocesa el texto de entrada usando el **tokenizer de BETO**.
- Realiza una inferencia con el modelo fine-tuned.
- Convierte los logits en **probabilidades** mediante la función softmax.
- Devuelve tanto la **categoría predicha** como un diccionario con las **probabilidades para cada clase**.

Esta función será utilizada para clasificar nuevos documentos empresariales en tiempo real durante las pruebas de validación.

In [5]:
def classify_document(text: str):
    """
    Clasifica un texto usando el modelo BETO fine-tuned.
    Retorna: (clase_predicha, probabilidades_por_clase)
    """
    inputs = tokenizer_beto(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    outputs = model_beto(**inputs)
    probs = outputs.logits.softmax(dim=-1).detach().numpy()[0]
    pred_id = probs.argmax()
    pred_label = id2label[pred_id]
    return pred_label, {id2label[i]: prob for i, prob in enumerate(probs)}

## Función de búsqueda de documentos similares

Define la función `find_similar_documents(query_text, category, threshold, top_k)`, que permite identificar documentos del corpus empresarial que son semánticamente similares a un texto de consulta, utilizando los **embeddings generados con Sentence-BERT**.

### Características clave:
- **Búsqueda contextual**: compara el significado del texto, no solo coincidencias léxicas.
- **Filtrado opcional por categoría**: mejora la relevancia al limitar la comparación a documentos del mismo tipo (por ejemplo, solo *Cotizaciones*).
- **Umbral de similitud ajustable**: permite controlar la sensibilidad de la detección de duplicados o documentos relacionados.
- **Resultado estructurado**: devuelve un `DataFrame` con el nombre del archivo, su categoría y el grado de similitud (0–1).

Esta función es fundamental para la validación de la funcionalidad de **detección de duplicados** (RF5) y puede usarse tanto en pruebas manuales como en la futura API REST.

In [6]:
def find_similar_documents(query_text: str, category: str = None, threshold: float = 0.90, top_k: int = 5):
    """
    Busca documentos similares en el corpus utilizando Sentence-BERT.
    
    Parámetros:
      - query_text: texto a comparar
      - category: si se especifica, solo busca en esa categoría
      - threshold: similitud mínima para considerarlo duplicado
      - top_k: número máximo de resultados a devolver
    
    Retorna: DataFrame con columnas: filename, category, similarity
    """
    # Generar embedding del texto de consulta
    query_emb = sbert.encode([query_text])
    
    # Filtrar por categoría si se indica
    if category:
        mask = df_corpus["category"] == category
        indices = df_corpus[mask].index.tolist()
        ref_embeddings = embeddings[indices]
        ref_df = df_corpus.loc[indices].reset_index(drop=True)
    else:
        ref_embeddings = embeddings
        ref_df = df_corpus
        indices = list(range(len(df_corpus)))
    
    # Calcular similitudes
    sims = cosine_similarity(query_emb, ref_embeddings)[0]
    
    # Filtrar por umbral y ordenar
    results = []
    for i, sim in enumerate(sims):
        if sim >= threshold:
            results.append({
                "filename": ref_df.iloc[i]["filename"],
                "category": ref_df.iloc[i]["category"],
                "similarity": float(sim)
            })
    
    # Ordenar por similitud descendente
    results = sorted(results, key=lambda x: x["similarity"], reverse=True)[:top_k]
    return pd.DataFrame(results)

In [7]:
# Ejemplo de texto de prueba (puedes reemplazarlo)
texto_prueba = """
Estimado cliente, le informamos que su cotización número CA1234 ha sido aprobada...
"""

clase, probs = classify_document(texto_prueba)
print("🔍 Clasificación:")
print(f" - Clase predicha: {clase}")
print(" - Probabilidades:")
for cat, p in probs.items():
    print(f"   {cat}: {p:.4f}")

🔍 Clasificación:
 - Clase predicha: Correspondencia administrativa
 - Probabilidades:
   Contratos: 0.0139
   Correos electrónicos: 0.0342
   Correspondencia administrativa: 0.8231
   Cotizaciones: 0.0460
   Documentos fiscales: 0.0260
   Recursos Humanos: 0.0568


## Prueba integrada: clasificación de un documento aleatorio

Esta celda selecciona un documento al azar del corpus anonimizado, lo clasifica usando el modelo **BETO fine-tuned** y compara la predicción con la categoría real del documento.

Este tipo de prueba permite:
- Validar visualmente el comportamiento del modelo con datos reales.
- Detectar posibles errores o ambigüedades en documentos límite.
- Verificar que la inferencia se alinea con las expectativas del dominio empresarial guatemalteco.

In [12]:
import random

# Cargar el corpus anonimizado (asegúrate de que CORPUS_ANON_PATH esté definido)
df_anon = pd.read_csv(CORPUS_ANON_PATH)

# Seleccionar un documento aleatorio
random_idx = random.randint(0, len(df_anon) - 1)
random_doc = df_anon.iloc[random_idx]

real_category = random_doc["category"]
text_to_classify = random_doc["text"]

# Clasificar con el modelo BETO
predicted_category, class_probs = classify_document(text_to_classify)

# Evaluar acierto
is_correct = (predicted_category == real_category)

# Mostrar resultados
print("📄 Documento aleatorio del corpus anonimizado:")
print(f"- Archivo: {random_doc['filename']}")
print(f"- Categoría real: {real_category}")
print(f"- Categoría predicha: {predicted_category}")
print(f"- {'✅ ¡Predicción correcta!' if is_correct else '❌ Predicción incorrecta'}")

print("\n📝 Contenido (primeros 500 caracteres):")
print(text_to_classify[:500] + ("..." if len(text_to_classify) > 500 else ""))

print("\n📊 Probabilidades por categoría:")
for cat, prob in class_probs.items():
    marker = " ← predicha" if cat == predicted_category else ""
    print(f"  {cat}: {prob:.4f}{marker}")

📄 Documento aleatorio del corpus anonimizado:
- Archivo: Permiso para Asistir a Graduacion de un Familiar.txt
- Categoría real: Recursos Humanos
- Categoría predicha: Recursos Humanos
- ✅ ¡Predicción correcta!

📝 Contenido (primeros 500 caracteres):
GUATETALENTS
SOLUCIONES EN RRHH
FECHA DE SOLICITUD: 11/06/2024
NOMBRE DE EMPLEADO: Carlos Andy Tiniguar
CENTRO DE TRABAJO: Europlaza NOMBRE DE JEFE IMEDIATO: Rolando Valdes

PUESTO DE EMPLEADO: Programador Junior CODIGO DE EMPLEADO:

[_]PeRMIsO CON GOCE DE SUELDO [_]matrimonio [_]NacimienTo DE HUO
[__]PERMISO SIN GOCE DE SUELDO [__]AMONESTACION VERBAL [DX] vacaciones
[_]LLtEGaDA TaRDIA [__]AMONESTACION ESCRITA [_]Feuiciracion

[_]FALTA INJUSTIFICADA [__|susPension [_]PRemiacion

[_]ABANDONO DE L...

📊 Probabilidades por categoría:
  Contratos: 0.0011
  Correos electrónicos: 0.0022
  Correspondencia administrativa: 0.0022
  Cotizaciones: 0.0016
  Documentos fiscales: 0.0025
  Recursos Humanos: 0.9904 ← predicha


## Prueba múltiple: evaluación rápida con varios documentos aleatorios

Esta celda repite la clasificación sobre **varios documentos seleccionados al azar** (por defecto, 5) y muestra:
- La categoría real vs. la predicha en cada caso.
- Un resumen final con el número y porcentaje de aciertos.

Es útil para obtener una **evaluación cualitativa rápida** sin ejecutar una métrica completa sobre todo el conjunto de datos. Ideal para sesiones de validación interactiva con datos reales del contexto empresarial guatemalteco.

In [13]:
import random

# Configuración
num_samples = 5
correct_predictions = 0
results = []

# Cargar el corpus anonimizado
df_anon = pd.read_csv(CORPUS_ANON_PATH)

print(f"🔍 Clasificando {num_samples} documentos aleatorios del corpus...\n")

for i in range(num_samples):
    # Seleccionar documento aleatorio
    idx = random.randint(0, len(df_anon) - 1)
    doc = df_anon.iloc[idx]
    
    real_cat = doc["category"]
    text = doc["text"]
    
    # Clasificar
    pred_cat, _ = classify_document(text)
    
    # Evaluar
    is_correct = (pred_cat == real_cat)
    correct_predictions += int(is_correct)
    
    # Guardar para resumen
    results.append({
        "filename": doc["filename"],
        "real": real_cat,
        "predicted": pred_cat,
        "correct": is_correct
    })
    
    # Mostrar resultado individual
    status = "✅" if is_correct else "❌"
    print(f"{status} Ejemplo {i+1}:")
    print(f"   - Real: {real_cat}")
    print(f"   - Predicho: {pred_cat}")
    print(f"   - Archivo: {doc['filename']}")
    print()

# Resumen final
accuracy = correct_predictions / num_samples
print("=" * 50)
print(f"📈 Resultado de la prueba (n={num_samples}):")
print(f" - Aciertos: {correct_predictions}/{num_samples}")
print(f" - Precisión estimada: {accuracy:.2%}")
print("=" * 50)

🔍 Clasificando 5 documentos aleatorios del corpus...

✅ Ejemplo 1:
   - Real: Documentos fiscales
   - Predicho: Documentos fiscales
   - Archivo: BILL.50015076.20250901.3866851.txt

✅ Ejemplo 2:
   - Real: Recursos Humanos
   - Predicho: Recursos Humanos
   - Archivo: Vacaciones Septiembre - Kevin García.txt

✅ Ejemplo 3:
   - Real: Contratos
   - Predicho: Contratos
   - Archivo: CC7345-32853932.txt

✅ Ejemplo 4:
   - Real: Correspondencia administrativa
   - Predicho: Correspondencia administrativa
   - Archivo: Carta de Aceptación - Celtech, S.A. Deal 1011674 y CC 1150871 Parcial I.txt

✅ Ejemplo 5:
   - Real: Recursos Humanos
   - Predicho: Recursos Humanos
   - Archivo: Vacaciones Semana Santa 2024 - Carlos Andy Tiniguar.txt

📈 Resultado de la prueba (n=5):
 - Aciertos: 5/5
 - Precisión estimada: 100.00%


## Evaluación de duplicados: un documento aleatorio

Esta celda selecciona un documento al azar del corpus anonimizado y utiliza **Sentence-BERT** para buscar documentos **altamente similares** dentro de la **misma categoría**.

Se muestra:
- El documento de consulta (fragmento).
- Su categoría real.
- Los documentos duplicados encontrados (nombre, categoría, similitud ≥ 0.85).

Este enfoque permite validar visualmente si la detección de duplicados captura **equivalencia semántica** (no solo coincidencia léxica), especialmente en formatos repetitivos como solicitudes de permiso o cotizaciones.

In [8]:
# Mismo texto de prueba
texto_prueba = """
Estimado cliente, le informamos que su cotización número CA1234 ha sido aprobada...
"""

# Detectar duplicados en todas las categorías
print("🔍 Buscando duplicados en todo el corpus...")
duplicados = find_similar_documents(texto_prueba, threshold=0.85, top_k=3)

if duplicados.empty:
    print("❌ No se encontraron documentos similares.")
else:
    print(duplicados.to_string(index=False))

# También puedes restringir por categoría (si ya la conoces o la predijiste)
print("\n🔍 Buscando duplicados solo en 'Cotizaciones'...")
duplicados_cat = find_similar_documents(
    texto_prueba, 
    category="Cotizaciones", 
    threshold=0.85, 
    top_k=3
)
if duplicados_cat.empty:
    print("❌ No se encontraron duplicados en Cotizaciones.")
else:
    print(duplicados_cat.to_string(index=False))

🔍 Buscando duplicados en todo el corpus...
❌ No se encontraron documentos similares.

🔍 Buscando duplicados solo en 'Cotizaciones'...
❌ No se encontraron duplicados en Cotizaciones.


In [14]:
import random

# Seleccionar documento aleatorio
df_anon = pd.read_csv(CORPUS_ANON_PATH)
random_idx = random.randint(0, len(df_anon) - 1)
random_doc = df_anon.iloc[random_idx]

query_text = random_doc["text"]
true_category = random_doc["category"]
filename = random_doc["filename"]

print("📄 Documento de consulta:")
print(f"- Archivo: {filename}")
print(f"- Categoría: {true_category}")
print("\n📝 Contenido (primeros 400 caracteres):")
print(query_text[:400] + ("..." if len(query_text) > 400 else ""))

# Buscar duplicados en la misma categoría
print(f"\n🔍 Buscando duplicados en '{true_category}'...")
duplicados = find_similar_documents(
    query_text=query_text,
    category=true_category,
    threshold=0.85,
    top_k=5
)

if duplicados.empty:
    print("✅ No se encontraron duplicados (similitud ≥ 0.85).")
else:
    print(f"⚠️ Se encontraron {len(duplicados)} documento(s) similar(es):")
    for _, row in duplicados.iterrows():
        print(f"  - {row['filename']} → sim: {row['similarity']:.3f}")

📄 Documento de consulta:
- Archivo: 20251204_161036_Re Solicitud de usuario citrix, y forticlient.txt
- Categoría: Correos electrónicos

📝 Contenido (primeros 400 caracteres):
Asunto: Re: Solicitud de usuario citrix, y forticlient

Fecha: 04/04/2020 10:20 AM
De: Ronald Estuardo Contreras Ciraiz <rcontreras@celtech.com.gt>

Hola buenos dias

Consultando nuevamente si hay algun avance de lo solicitado anteriormente.
Quedo atento a sus comentarios y su pronta respuesta

Cualquier duda o consulta a la orden

Sls.

El mar., 17 de mar. de 2020 a la(s) 09:08, Ronald Estuardo C...

🔍 Buscando duplicados en 'Correos electrónicos'...
✅ No se encontraron duplicados (similitud ≥ 0.85).


## Evaluación múltiple: detección de duplicados en varios documentos

Esta celda repite la búsqueda de duplicados en **varios documentos seleccionados al azar** (por defecto, 4).  
Para cada uno, muestra si se encontraron candidatos a duplicado.

Este análisis rápido ayuda a:
- Verificar la **sensibilidad** del umbral de similitud.
- Identificar documentos "plantilla" que generan muchos duplicados (ej. formatos de RRHH).
- Detectar posibles falsos positivos o falsos negativos en contextos reales.

> **Nota**: Se restringe la búsqueda a la categoría correcta, tal como se hizo durante la generación del conjunto de duplicados en tu tesis.

In [15]:
import random

num_samples = 4
duplicated_count = 0

df_anon = pd.read_csv(CORPUS_ANON_PATH)

print(f"🔍 Analizando {num_samples} documentos aleatorios para detección de duplicados...\n")

for i in range(num_samples):
    idx = random.randint(0, len(df_anon) - 1)
    doc = df_anon.iloc[idx]
    
    text = doc["text"]
    cat = doc["category"]
    fname = doc["filename"]
    
    # Buscar duplicados en su categoría
    dups = find_similar_documents(
        query_text=text,
        category=cat,
        threshold=0.85,
        top_k=3
    )
    
    has_dup = not dups.empty
    if has_dup:
        duplicated_count += 1
    
    status = "⚠️ Duplicados encontrados" if has_dup else "✅ Único"
    print(f"{status} | {fname} ({cat})")
    if has_dup:
        for _, r in dups.iterrows():
            print(f"    → {r['filename']} (sim: {r['similarity']:.3f})")
    print()

# Resumen
print("=" * 50)
print(f"📈 Resultado (n={num_samples}):")
print(f" - Documentos con duplicados: {duplicated_count}")
print(f" - Documentos únicos: {num_samples - duplicated_count}")
print(f" - Tasa empírica de duplicación: {duplicated_count / num_samples:.2%}")
print("=" * 50)

🔍 Analizando 4 documentos aleatorios para detección de duplicados...

✅ Único | 20251204_161057_Envió de Correspondencia Semanal.txt (Correos electrónicos)

✅ Único | Presentacion PBX Virtual UCaaS.txt (Correspondencia administrativa)

✅ Único | Propuesta Internet Corporativo CELTECH SOCIEDAD ANONIMA.txt (Cotizaciones)

⚠️ Duplicados encontrados | 20251204_160746_Re Solicitud de Cambio de HH 902 y 903.txt (Correos electrónicos)
    → 20251204_160746_Re Solicitud de Cambio de HH 902 y 903.txt (sim: 0.855)

📈 Resultado (n=4):
 - Documentos con duplicados: 1
 - Documentos únicos: 3
 - Tasa empírica de duplicación: 25.00%


In [9]:
def test_pipeline(text: str):
    print("="*60)
    print("📝 Documento de entrada:")
    print(text[:300] + ("..." if len(text) > 300 else ""))
    print("\n⚙️ Etapa 1: Clasificación...")
    categoria, _ = classify_document(text)
    print(f" ➤ Categoría predicha: {categoria}")
    
    print("\n⚙️ Etapa 2: Búsqueda de duplicados en esa categoría...")
    duplicados = find_similar_documents(text, category=categoria, threshold=0.85, top_k=2)
    if not duplicados.empty:
        print(" ➤ Posibles duplicados encontrados:")
        for _, row in duplicados.iterrows():
            print(f"   - {row['filename']} ({row['category']}) → sim: {row['similarity']:.3f}")
    else:
        print(" ➤ ✅ No se encontraron duplicados.")
    print("="*60)

# Ejemplo de uso
test_pipeline("Solicitud de permiso para el día 10 de mayo de 2024 debido a emergencia familiar.")

📝 Documento de entrada:
Solicitud de permiso para el día 10 de mayo de 2024 debido a emergencia familiar.

⚙️ Etapa 1: Clasificación...
 ➤ Categoría predicha: Correspondencia administrativa

⚙️ Etapa 2: Búsqueda de duplicados en esa categoría...
 ➤ ✅ No se encontraron duplicados.


## Metadatos del modelo BETO fine-tuned

Esta celda muestra información detallada del modelo **BETO fine-tuned**, incluyendo:
- Nombre del modelo base.
- Arquitectura (`BertForSequenceClassification`).
- Número de etiquetas y mapeo `id ↔ categoría`.
- Configuración del tokenizador (longitud máxima, tokens especiales, vocabulario).
- Ruta de guardado.

Estos metadatos garantizan la **trazabilidad** del modelo y son esenciales para reproducibilidad, auditoría y despliegue en producción.

In [16]:
# === Metadatos del modelo BETO ===
print("🔍 Información del modelo BETO fine-tuned:\n")

# Modelo base
print(f"• Modelo base: {model_beto.config._name_or_path}")

# Arquitectura
print(f"• Arquitectura: {model_beto.__class__.__name__}")

# Número de clases
print(f"• Número de categorías: {model_beto.config.num_labels}")

# Mapeo id2label
print("• Mapeo id → categoría:")
for idx, label in model_beto.config.id2label.items():
    print(f"    {idx}: {label}")

# Tokenizador
print(f"\n• Tokenizador:")
print(f"  - Vocabulario: {len(tokenizer_beto.vocab):,} tokens")
print(f"  - Longitud máxima por defecto: {tokenizer_beto.model_max_length}")
print(f"  - Tokens especiales: {tokenizer_beto.special_tokens_map}")

# Ruta
print(f"\n• Ruta cargada: {BETO_MODEL_PATH.resolve()}")

🔍 Información del modelo BETO fine-tuned:

• Modelo base: models\modelo_beto_finetuned_v1
• Arquitectura: BertForSequenceClassification
• Número de categorías: 6
• Mapeo id → categoría:
    0: Contratos
    1: Correos electrónicos
    2: Correspondencia administrativa
    3: Cotizaciones
    4: Documentos fiscales
    5: Recursos Humanos

• Tokenizador:
  - Vocabulario: 31,002 tokens
  - Longitud máxima por defecto: 512
  - Tokens especiales: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}

• Ruta cargada: C:\Users\Rolando\TAREAS MIA\Tesis\models\modelo_beto_finetuned_v1


## Metadatos del modelo Sentence-BERT

Esta celda muestra información técnica del modelo **Sentence-BERT** utilizado para generar embeddings semánticos, incluyendo:
- Nombre del modelo (`paraphrase-multilingual-mpnet-base-v2`).
- Dimensión del embedding (768).
- Número de documentos en el corpus de referencia.
- Estadísticas de los embeddings precomputados (forma, rango, tipo de dato).

Esta información es clave para entender el **alcance y límites** del sistema de detección de duplicados, y para validar que los artefactos están alineados.

In [18]:
# === Metadatos del modelo Sentence-BERT ===
print("🔍 Información del modelo Sentence-BERT:\n")

# Nombre del modelo
sbert_config_path = SBERT_MODEL_PATH / "config_sentence_transformers.json"
if sbert_config_path.exists():
    import json
    with open(sbert_config_path, "r", encoding="utf-8") as f:
        sbert_config = json.load(f)
    print(f"• Modelo: {sbert_config.get('_name_or_path', 'No disponible')}")
else:
    print("• Modelo: paraphrase-multilingual-mpnet-base-v2 (nombre predeterminado)")

# Dimensión del embedding
embedding_dim = embeddings.shape[1] if embeddings.ndim == 2 else "Desconocido"
print(f"• Dimensión del embedding: {embedding_dim}")

# Corpus de referencia
print(f"• Documentos en el corpus: {len(df_corpus)}")
print(f"• Categorías en el corpus: {sorted(df_corpus['category'].unique())}")

# Embeddings
print(f"\n• Embeddings precomputados:")
print(f"  - Forma: {embeddings.shape}")
print(f"  - Tipo de dato: {embeddings.dtype}")
print(f"  - Rango de valores: [{embeddings.min():.4f}, {embeddings.max():.4f}]")

# Ruta
print(f"\n• Ruta del modelo: {SBERT_MODEL_PATH.resolve()}")

🔍 Información del modelo Sentence-BERT:

• Modelo: No disponible
• Dimensión del embedding: 768
• Documentos en el corpus: 906
• Categorías en el corpus: ['Contratos', 'Correos electrónicos', 'Correspondencia administrativa', 'Cotizaciones', 'Documentos fiscales', 'Recursos Humanos']

• Embeddings precomputados:
  - Forma: (906, 768)
  - Tipo de dato: float32
  - Rango de valores: [-0.4475, 0.5106]

• Ruta del modelo: C:\Users\Rolando\TAREAS MIA\Tesis\modelo_sbert_final
